# HR Salary Analysis — Class Recap
## What we covered last class and what comes next

---

## ✅ Questions we already answered

**Section 1 — First Look**
- How many salary records are there?
- How many unique workers?
- What years does the data cover?
- How many position levels and departments exist?

**Section 2 — First Questions (no join needed)**
- Who has the highest net salary? → `sort_values` and `nlargest`
- Who gets the highest bonus?
- Who has been at the company the longest?

**Section 3 — Why Comparisons Need Context**
- What is the overall average salary? *(and why it is misleading)*
- What is the average salary per position level? *(the fair comparison)*

**Section 4 — Salary by Department**
- Which department has the highest average salary?
- Is that difference explained by position level? *(confounding variable)*
- Fair comparison: Analysts only across departments
- Salary by area (Technology, Business, Operations, Commercial)

**Section 5 — Salary Evolution Over Time**
- How did average salary change year by year?
- Senior Analysts only — fair time comparison
- Line chart per department — which team grew fastest?

---

## 🔲 Questions still to answer today

**Section 6 — Career Progression**
- Which workers got promoted?
- Who got promoted the most times?
- What does a specific worker's career path look like?
- How to see multiple workers or a range of IDs at once?

**Section 7 — Tenure Analysis**
- Do workers who stay longer earn more?
- Is the salary difference explained by position level or genuinely by tenure?
- Fair comparison: same level, different tenure group

**Section 8 — Challenges**
- Challenge 1: Which worker earned the most in total across all years?
- Challenge 2: Which department pays the highest average bonus?
- Challenge 3: How many workers were hired each year?

---

> *Run the setup cell first — it reloads all four files and recreates the joined table.*

# HR Salary Analysis
### A dataset you have never seen before — explore it yourself

You have four files:
- `hr_salary_records.csv` — the fact table (one row per worker per year)
- `hr_workers.csv` — worker personal and contract info
- `hr_positions.csv` — position levels and salary bands
- `hr_departments.csv` — department and area info

**How we work today:**
- We join dimension tables only when we need them to answer a question
- Every calculation gets its own variable
- Every print is a separate step
- When comparing salaries — always ask: *does this comparison make sense?*

# 0. Setup

In [ ]:
# ── Run this first — reloads everything from last class ────────

# ── Google Colab ───────────────────────────────────────────
# from google.colab import files
# files.upload()   # upload all four CSV files

# ── VS Code ─────────────────────────────────────────────────
# Make sure all four CSV files are in the same folder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load all four files
salary  = pd.read_csv("hr_salary_records.csv")
workers = pd.read_csv("hr_workers.csv")
pos     = pd.read_csv("hr_positions.csv")
depts   = pd.read_csv("hr_departments.csv")

# Recreate joins from last class
hr = salary.merge(pos,   on="position_id",   how="left")
hr = hr.merge(depts,     on="department_id",  how="left")

# Calculate tenure
workers["admission_date"]   = pd.to_datetime(workers["admission_date"])
workers["years_at_company"] = ((pd.Timestamp("today") - workers["admission_date"]).dt.days / 365).round(1)

print("Everything loaded and joined!")
print("salary shape: ", salary.shape)
print("hr shape:     ", hr.shape)
print()
print("Columns in hr:")
print(hr.columns.tolist())

Saving hr_departments.csv to hr_departments.csv
Saving hr_positions.csv to hr_positions.csv
Saving hr_salary_records.csv to hr_salary_records.csv
Saving hr_workers.csv to hr_workers.csv
salary_records shape:  (151, 11)
workers shape:         (40, 9)
positions shape:       (12, 5)
departments shape:     (8, 4)


# 1. First Look — 10 Minutes, You Work Alone
No hints yet. Use the four commands you know.
Look at each table before doing anything else.

## 1.1 — Explore the fact table

In [ ]:
salary.shape

(151, 11)

In [ ]:
salary.head(10)

,record_id,worker_id,department_id,position_id,year,base_salary,bonus,additional_allowance,gross_total,taxes,net_salary
0,R0001,W001,D04,P08,2022,69500.0,7200.0,6000,82700.0,31426.0,51274.0
1,R0002,W001,D04,P08,2023,72500.0,7200.0,7200,86900.0,33022.0,53878.0
2,R0003,W001,D04,P08,2024,74800.0,4300.0,9600,88700.0,33706.0,54994.0
3,R0004,W002,D08,P02,2023,15300.0,0.0,2400,17700.0,2478.0,15222.0
4,R0005,W002,D08,P02,2024,16000.0,0.0,2400,18400.0,2576.0,15824.0
5,R0006,W003,D02,P02,2024,15200.0,0.0,2400,17600.0,2464.0,15136.0
6,R0007,W004,D07,P06,2020,58900.0,2500.0,1200,62600.0,20032.0,42568.0
7,R0008,W004,D07,P06,2021,60700.0,1900.0,2400,65000.0,20800.0,44200.0
8,R0009,W005,D06,P07,2019,54800.0,3700.0,2400,60900.0,19488.0,41412.0
9,R0010,W005,D06,P07,2020,56400.0,3500.0,2400,62300.0,19936.0,42364.0


In [ ]:
salary.dtypes

,0
record_id,object
worker_id,object
department_id,object
position_id,object
year,int64
base_salary,float64
bonus,float64
additional_allowance,int64
gross_total,float64
taxes,float64


In [ ]:
salary.describe()

,year,base_salary,bonus,additional_allowance,gross_total,taxes,net_salary
count,151.000000,151.000000,151.000000,151.000000,151.000000,151.000000,151.000000
mean,2021.860927,52211.920530,3169.536424,3178.807947,58560.264901,20150.874172,38409.390728
std,1.705050,27850.701792,4160.704962,4104.511490,34590.376464,16001.982546,18739.500013
min,2018.000000,12700.000000,0.000000,0.000000,12700.000000,1778.000000,10922.000000
25%,2021.000000,30300.000000,0.000000,600.000000,31150.000000,7476.000000,23674.000000
50%,2022.000000,53500.000000,2000.000000,1200.000000,58000.000000,18560.000000,39440.000000
75%,2023.000000,68950.000000,3900.000000,4200.000000,76650.000000,29127.000000,48941.000000
max,2024.000000,117700.000000,23700.000000,18000.000000,149800.000000,62916.000000,86884.000000


## 1.2 — Explore the workers table

In [ ]:
workers.head(10)

,worker_id,full_name,gender,nationality,date_of_birth,admission_date,end_date,is_active,department_id
0,W001,Alice Wagner,Female,Italian,1976-06-03,2022-05-17,2024-04-18,False,D04
1,W002,Tina Becker,Female,French,1998-10-19,2023-03-12,NaN,True,D08
2,W003,Olivia Richter,Female,Brazilian,2000-07-07,2024-10-11,NaN,True,D02
3,W004,Sofia Braun,Female,Indian,1988-06-14,2020-02-05,2021-11-21,False,D07
4,W005,Kate Hoffmann,Female,Brazilian,1978-01-13,2019-03-17,2021-04-07,False,D06
5,W006,Yara Schwarz,Female,Italian,1989-03-14,2018-08-05,NaN,True,D03
6,W007,Lukas Hartmann,Male,German,1982-02-15,2018-07-21,NaN,True,D02
7,W008,Max Hoffmann,Male,Spanish,1992-01-03,2024-09-01,NaN,True,D04
8,W009,Bob Krüger,Male,German,1990-08-07,2024-08-01,NaN,True,D07
9,W010,Uma Lange,Female,Indian,1987-01-13,2021-01-22,NaN,True,D05


In [ ]:
# How many active vs left?
status_counts = workers["is_active"].value_counts()
print(status_counts)
# True  = active
# False = left

is_active
True     32
False     8
Name: count, dtype: int64


## 1.3 — Explore positions and departments

In [ ]:
print(pos.to_string(index=False))

position_id   position_name  level  salary_band_min  salary_band_max
        P01          Intern      1            12000            18000
        P02 Working Student      1            14000            20000
        P03       Assistant      2            28000            36000
        P04  Junior Analyst      3            36000            46000
        P05         Analyst      4            44000            56000
        P06  Senior Analyst      5            54000            68000
        P07     Coordinator      5            52000            66000
        P08         Manager      6            66000            88000
        P09  Senior Manager      7            82000           105000
        P10        Director      8           100000           130000
        P11              VP      9           125000           160000
        P12         C-Level     10           150000           220000


In [ ]:
print(depts.to_string(index=False))

department_id    department_name       area cost_center
          D01            Finance   Business      CC-101
          D02    Human Resources   Business      CC-102
          D03          Logistics Operations      CC-103
          D04              Sales Commercial      CC-104
          D05         Operations Operations      CC-105
          D06                 IT Technology      CC-106
          D07 Project Management   Business      CC-107
          D08       Data Science Technology      CC-108


How many salary records in total?


In [ ]:
# How many salary records in total?
# Hint: len(df)
total_records = len(salary)
print("Total records:", total_records)



Total records: 151


# How many unique workers?

In [ ]:
# Hint: df["col"].nunique()
unique_workers = salary["worker_id"].nunique()
print("Unique workers:", unique_workers)



Unique workers: 40


# What years does the data cover?

In [ ]:
# Hint: df["col"].min() and df["col"].max()
first_year = salary["year"].min()
last_year  = salary["year"].max()
print("Years covered:", first_year, "to", last_year)


Years covered: 2018 to 2024


# How many position levels?

In [ ]:


num_levels = pos["level"].nunique()
print("Position levels:", num_levels)



Position levels: 10


# How many departments?

In [ ]:

num_depts = depts["department_id"].nunique()
print("Departments:", num_depts)


Departments: 8


# What are the areas?

In [ ]:


areas = depts["area"].nunique()
print("Areas:", areas)

print(depts["area"])

Areas: 4
0      Business
1      Business
2    Operations
3    Commercial
4    Operations
5    Technology
6      Business
7    Technology
Name: area, dtype: object


# 2. First Questions — No Join Needed Yet
Before joining anything — let's answer the easy questions directly from the fact table.
These are the questions everyone asks first when they see HR data.

## 2.1 — Who has the highest net salary?
We will show two ways to get the same answer.
Both are correct — choose the one that feels more natural to you.

In [ ]:
salary.head(2)

,record_id,worker_id,department_id,position_id,year,base_salary,bonus,additional_allowance,gross_total,taxes,net_salary
0,R0001,W001,D04,P08,2022,69500.0,7200.0,6000,82700.0,31426.0,51274.0
1,R0002,W001,D04,P08,2023,72500.0,7200.0,7200,86900.0,33022.0,53878.0


In [ ]:
# Way 1 — sort_values and head
# Hint: df.sort_values("col", ascending=False).head(n)
top_earner = salary.sort_values("net_salary", ascending=False).head(5)

print("Highest net salary:")
print(top_earner)

Highest net salary:
    record_id worker_id department_id position_id  year  base_salary    bonus  \
90      R0091      W025           D02         P10  2022     110100.0  21700.0   
91      R0092      W025           D02         P10  2023     112700.0  19000.0   
109     R0110      W029           D03         P10  2023     107000.0  18400.0   
89      R0090      W025           D02         P10  2021     106000.0  23700.0   
48      R0049      W014           D02         P10  2024     109300.0  11400.0   

     additional_allowance  gross_total    taxes  net_salary  
90                  18000     149800.0  62916.0     86884.0  
91                  12000     143700.0  60354.0     83346.0  
109                 18000     143400.0  60228.0     83172.0  
89                  12000     141700.0  59514.0     82186.0  
48                  18000     138700.0  58254.0     80446.0  


In [ ]:
# Way 2 — nlargest
# Hint: df.nlargest(n, "col")
top_earner2 = salary.nlargest(5, "net_salary")

print("Highest net salary (nlargest):")
print(top_earner2[["worker_id", "year", "net_salary"]].to_string(index=False))

# Same result — two different ways to get there
# sort_values is more flexible (can sort by multiple columns)
# nlargest is shorter when you only need the top rows

Highest net salary (nlargest):
worker_id  year  net_salary
     W025  2022     86884.0
     W025  2023     83346.0
     W029  2023     83172.0
     W025  2021     82186.0
     W014  2024     80446.0


### Your turn — top 5 earners

In [ ]:
# Hint: df.nlargest(n, "col")
top5_earners = salary.nlargest(5, "net_salary")

print("Top 5 net salaries:")
print(top5_earners[["worker_id", "year", "net_salary"]].to_string(index=False))

Top 5 net salaries:
worker_id  year  net_salary
     W025  2022     86884.0
     W025  2023     83346.0
     W029  2023     83172.0
     W025  2021     82186.0
     W014  2024     80446.0


## 2.2 — Who gets the highest bonus?

In [ ]:
# Example — highest bonus ever using sort_values
top_bonus = salary.sort_values("bonus", ascending=False).head(1)

print("Highest bonus:")
print(top_bonus[["worker_id", "year", "bonus"]].to_string(index=False))

Highest bonus:
worker_id  year   bonus
     W025  2021 23700.0


### Your turn — top 5 bonuses using nlargest

In [ ]:
# Way 1 — nlargest
# Hint: df.nlargest(n, "col")
top_bonus = salary.nlargest(10,"bonus")

print("bonus:")
print(top_bonus)
print()



bonus:
    record_id worker_id department_id position_id  year  base_salary    bonus  \
89      R0090      W025           D02         P10  2021     106000.0  23700.0   
90      R0091      W025           D02         P10  2022     110100.0  21700.0   
91      R0092      W025           D02         P10  2023     112700.0  19000.0   
109     R0110      W029           D03         P10  2023     107000.0  18400.0   
47      R0048      W014           D02         P10  2023     106000.0  17800.0   
48      R0049      W014           D02         P10  2024     109300.0  11400.0   
24      R0025      W007           D02         P08  2024      87000.0   9300.0   
23      R0024      W007           D02         P08  2023      83100.0   9200.0   
95      R0096      W026           D04         P08  2021      79400.0   9200.0   
92      R0093      W025           D02         P10  2024     117700.0   9000.0   

     additional_allowance  gross_total    taxes  net_salary  
89                  12000     141700.0 

In [ ]:
# Way 2 — sort_values and head
# Hint: df.sort_values(ascending=False).head(n)
top_bonus2 = salary.sort_values("bonus",ascending=False).head(10)
print(top_bonus2)

    record_id worker_id department_id position_id  year  base_salary    bonus  \
89      R0090      W025           D02         P10  2021     106000.0  23700.0   
90      R0091      W025           D02         P10  2022     110100.0  21700.0   
91      R0092      W025           D02         P10  2023     112700.0  19000.0   
109     R0110      W029           D03         P10  2023     107000.0  18400.0   
47      R0048      W014           D02         P10  2023     106000.0  17800.0   
48      R0049      W014           D02         P10  2024     109300.0  11400.0   
24      R0025      W007           D02         P08  2024      87000.0   9300.0   
23      R0024      W007           D02         P08  2023      83100.0   9200.0   
95      R0096      W026           D04         P08  2021      79400.0   9200.0   
92      R0093      W025           D02         P10  2024     117700.0   9000.0   

     additional_allowance  gross_total    taxes  net_salary  
89                  12000     141700.0  59514.

### Notice something?
The worker_id tells us who — but not their name.
We need to join the workers table to get the name.
We will do that in the next section.

## 2.3 — Who has been at the company longest?

In [ ]:
workers.head()

,worker_id,full_name,gender,nationality,date_of_birth,admission_date,end_date,is_active,department_id,years_at_company
0,W001,Alice Wagner,Female,Italian,1976-06-03,2022-05-17,2024-04-18,False,D04,4.0
1,W002,Tina Becker,Female,French,1998-10-19,2023-03-12,NaN,True,D08,3.2
2,W003,Olivia Richter,Female,Brazilian,2000-07-07,2024-10-11,NaN,True,D02,1.6
3,W004,Sofia Braun,Female,Indian,1988-06-14,2020-02-05,2021-11-21,False,D07,6.3
4,W005,Kate Hoffmann,Female,Brazilian,1978-01-13,2019-03-17,2021-04-07,False,D06,7.2


In [ ]:
# Convert date and calculate tenure
workers["admission_date"]  = pd.to_datetime(workers["admission_date"])
workers["years_at_company"] = ((pd.Timestamp("today") - workers["admission_date"]).dt.days / 365).round(1)



In [ ]:
# Sort to find longest serving
# Hint: df.sort_values("col", ascending=False).head(5)
longest = workers.sort_values("years_at_company", ascending=False).head(5)

print("Longest serving workers:")
print(longest[["worker_id", "full_name", "admission_date", "years_at_company"]].to_string(index=False))

Longest serving workers:
worker_id      full_name admission_date  years_at_company
     W030    Mia Neumann     2018-05-05               8.1
     W036    Marco Lange     2018-04-24               8.1
     W007 Lukas Hartmann     2018-07-21               7.9
     W006   Yara Schwarz     2018-08-05               7.8
     W015   Jonas Wagner     2019-01-28               7.3


### Your turn — newest hires using nlargest or sort_values

In [ ]:
# Hint: df.nlargest finds largest — for newest hire use nlargest on admission_date
# Or: df.sort_values("col", ascending=False).head(5)
newest = workers.nlargest(5, "years_at_company")

print("Newest hires:")
print(newest[["full_name", "admission_date", "years_at_company"]].to_string(index=False))

Newest hires:
     full_name admission_date  years_at_company
   Mia Neumann     2018-05-05               8.1
   Marco Lange     2018-04-24               8.1
Lukas Hartmann     2018-07-21               7.9
  Yara Schwarz     2018-08-05               7.8
  Jonas Wagner     2019-01-28               7.3


# 3. Why Comparisons Need Context
We now want to understand salaries better.
But to do that we need the position name and level — which are in the positions table.

**This is the moment we join positions — because we need it now.**

> *"The average salary of the whole company tells you almost nothing.
> An Intern and a Director in the same average is not a useful number."*

In [ ]:
salary.head()

,record_id,worker_id,department_id,position_id,year,base_salary,bonus,additional_allowance,gross_total,taxes,net_salary
0,R0001,W001,D04,P08,2022,69500.0,7200.0,6000,82700.0,31426.0,51274.0
1,R0002,W001,D04,P08,2023,72500.0,7200.0,7200,86900.0,33022.0,53878.0
2,R0003,W001,D04,P08,2024,74800.0,4300.0,9600,88700.0,33706.0,54994.0
3,R0004,W002,D08,P02,2023,15300.0,0.0,2400,17700.0,2478.0,15222.0
4,R0005,W002,D08,P02,2024,16000.0,0.0,2400,18400.0,2576.0,15824.0


In [ ]:
pos.head()

,position_id,position_name,level,salary_band_min,salary_band_max
0,P01,Intern,1,12000,18000
1,P02,Working Student,1,14000,20000
2,P03,Assistant,2,28000,36000
3,P04,Junior Analyst,3,36000,46000
4,P05,Analyst,4,44000,56000


In [ ]:
# Join positions — we need position_name and level
# Hint: df.merge(other_df, on="shared_col", how="left")
hr = salary.merge(pos, on="position_id", how="left")

print(hr.head(5))

  record_id worker_id department_id position_id  year  base_salary   bonus  \
0     R0001      W001           D04         P08  2022      69500.0  7200.0   
1     R0002      W001           D04         P08  2023      72500.0  7200.0   
2     R0003      W001           D04         P08  2024      74800.0  4300.0   
3     R0004      W002           D08         P02  2023      15300.0     0.0   
4     R0005      W002           D08         P02  2024      16000.0     0.0   

   additional_allowance  gross_total    taxes  net_salary    position_name  \
0                  6000      82700.0  31426.0     51274.0          Manager   
1                  7200      86900.0  33022.0     53878.0          Manager   
2                  9600      88700.0  33706.0     54994.0          Manager   
3                  2400      17700.0   2478.0     15222.0  Working Student   
4                  2400      18400.0   2576.0     15824.0  Working Student   

   level  salary_band_min  salary_band_max  
0      6         

In [ ]:
# Overall average — misleading
avg_all = hr["net_salary"].mean()
print("Overall average net salary:", round(avg_all, 0), "euros")
print()

# Range — shows why the average is not useful
min_salary = hr["net_salary"].min()
max_salary = hr["net_salary"].max()
print("Minimum:", round(min_salary, 0), "euros")
print("Maximum:", round(max_salary, 0), "euros")
print()
print("The range is huge — Intern to C-Level in the same average makes no sense")

Overall average net salary: 38409.0 euros

Minimum: 10922.0 euros
Maximum: 86884.0 euros

The range is huge — Intern to C-Level in the same average makes no sense


In [ ]:
# The right approach — average per position level
avg_by_level = hr.groupby(["level", "position_name"])["net_salary"].mean().round(0)

print("Average net salary by position:")
print(avg_by_level.to_string())

Average net salary by position:
level  position_name  
1      Intern             11295.0
       Working Student    15062.0
2      Assistant          25688.0
3      Junior Analyst     29042.0
4      Analyst            36951.0
5      Coordinator        42400.0
       Senior Analyst     44001.0
6      Manager            56210.0
7      Senior Manager     62282.0
8      Director           81925.0


### Your turn — visualise average salary by position level

**Steps:**
1. groupby `level` and `position_name` together — calculate mean net_salary
2. reset_index so level becomes a regular column we can sort by
3. sort by `level` so positions appear in career order
4. plot a bar chart

In [ ]:
# Step 1 — calculate average salary per position
# Hint: df.groupby(["col1","col2"])["col"].mean().round(0)
avg_by_pos = hr.groupby(["_____", "____________"])["__________"].______().round(0)

In [ ]:
# Step 2 — reset index so level becomes a column we can sort
# Hint: df.reset_index()
avg_by_pos_df = avg_by_pos.____________()

In [ ]:

# Step 3 — sort by level so positions appear in career order
# Hint: df.sort_values("col")
avg_by_pos_sorted = avg_by_pos_df.sort_values("_____")

In [ ]:
# Step 4 — print
print(avg_by_pos_sorted[["position_name", "net_salary"]].to_string(index=False))

In [ ]:
# Step 3 — plot
plt.figure(figsize=(10, 4))
plt.bar(
    avg_by_pos_sorted["____________"],
    avg_by_pos_sorted["__________"],
    color="steelblue", edgecolor="white"
)

plt.title("Average Net Salary by Position")
plt.xlabel("Position")
plt.ylabel("Average Net Salary (euros)")
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.show()

### Your turn — top 3 highest paid positions using nlargest

In [ ]:
# Hint: df.nlargest(n, "col")
top3_positions = avg_by_pos_df.nlargest(___, "___________")

print("Top 3 highest paid positions:")
print(top3_positions[["position_name", "net_salary"]].to_string(index=False))

# 4. Salary by Department
Now we want to analyse by department — so we join departments now.

> *"If Data Science has more Senior Analysts and Finance has more Juniors,
> is the difference in average salary about the department or about the positions?
> This is called a confounding variable."*

In [ ]:
# Join departments — we need department_name and area now
# Hint: df.merge(other_df, on="shared_col", how="left")
hr = hr.merge(depts, on="___________", how="left")

print("After joining departments:")
print(hr[["worker_id", "position_name", "department_name", "area", "net_salary"]].head(5))

## 4.1 — Average salary by department

In [ ]:
# Step 1 — calculate
# Hint: df.groupby("col")["col"].mean().round(0)
avg_by_dept = hr.groupby("_______________")["__________"].______().round(0)



In [ ]:
# Step 2 — sort
avg_by_dept_sorted = avg_by_dept.sort_values(ascending=_______)



In [ ]:
# Step 3 — print
print("Average net salary by department:")
print(_____________)

In [ ]:
# Step 4 — plot
avg_by_dept_sorted.plot(kind="bar", color="steelblue")

plt.title("Average Net Salary by Department")
plt.xlabel("Department")
plt.ylabel("Average Net Salary (euros)")
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.show()

### Your turn — is this difference explained by position level?

In [ ]:
# Step 1 — average LEVEL per department
# Hint: df.groupby("col")["col"].mean().round(1)
avg_level_dept = hr.groupby("_______________")["_____"].______().round(1)



In [ ]:
# Step 2 — sort
avg_level_dept_sorted = ____________.________(ascending=_______)



In [ ]:
# Step 3 — print
print("Average position level by department:")
print(_______________)
print()
print("Does the level ranking match the salary ranking?")
print("If yes — the difference is about positions, not departments")

## 4.2 — Fair comparison: same level, different departments

In [ ]:
# Example — Analysts (level 4) across departments
# Same role, different teams — this is a fair comparison
analysts = hr[hr["level"] == 4]

analyst_by_dept = analysts.groupby("department_name")["net_salary"].mean().round(0)
print("Average net salary — Analysts only:")
print(analyst_by_dept.sort_values(ascending=False).to_string())

### Your turn — same comparison for Managers (level 6)

In [ ]:
# Step 1 — filter managers
# Hint: df[df["col"] == value]
managers = 



In [ ]:
# Step 2 — average by department
# Hint: df.groupby("col")["col"].mean().round(0)
manager_by_dept = 



In [ ]:
# Step 3 — sort
# Hint: df.sort_values(ascending=False)
manager_by_dept_sorted = )

In [ ]:
# Step 4 — print
print("Average net salary — Managers only:")
print(_______________)

## 4.3 — Salary by area

In [ ]:
# Step 1 — calculate
# Hint: df.groupby("col")["col"].mean().round(0)
avg_by_area = 



In [ ]:
# Step 2 — sort
avg_by_area_sorted =



In [ ]:
# Step 3 — print
print("Average net salary by area:")
print(_______________)

In [ ]:
# Step 4 — plot
___________.plot(kind="_________", color="coral")

plt.title("___")
plt.xlabel("Area")
plt.ylabel("___")
plt.xticks(rotation=___)

plt.tight_layout()
plt.show()

# 5. Salary Evolution Over Time
No new join needed here — year is already in the fact table.

> *"If we hired many junior workers in 2023, the overall average drops —
> not because salaries went down, but because the mix changed."*

In [ ]:
# Overall average per year
avg_by_year = hr.groupby("year")["net_salary"].mean().round(0)

print("Average net salary per year:")
print(avg_by_year.to_string())

In [ ]:
# Plot it
avg_by_year.plot(kind="bar", color="steelblue")

plt.title("Average Net Salary per Year")
plt.xlabel("Year")
plt.ylabel("Average Net Salary (euros)")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()
print()
print("Does this mean salaries went up or down?")
print("Or did the position mix change?")

### Your turn — Senior Analysts only (fair time comparison)

In [ ]:
# Step 1 — filter Senior Analysts
# Hint: df[df["col"] == "value"]
senior_analysts = hr[hr["____________"] == "Senior Analyst"]



In [ ]:
# Step 2 — average per year
# Hint: df.groupby("col")["col"].mean().round(0)
sa_by_year = ______________.__________("____")["__________"].______().round(0)



In [ ]:
# Step 3 — print
print("Senior Analyst — average net salary per year:")
print(_______________)

In [ ]:
# Line chart — shows growth trend clearly
________.plot(kind="___", marker="o", color="steelblue")

plt.title("Senior Analyst — Average Net Salary Over Time")
plt.xlabel("Year")
plt.ylabel("Average Net Salary (euros)")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

### Your turn — line chart per department (salary growth by team)

**Steps:**
1. groupby `year` and `department_name`, calculate mean net_salary
2. unstack to turn departments into columns
3. plot as a line chart

In [ ]:
# Step 1 — calculate
# Hint: df.groupby(["col1","col2"])["col"].mean().round(0).unstack()
avg_year_dept = hr.groupby(["____", "_______________"])["__________"].______().round(0)

# Step 2 — unstack turns departments into columns
avg_year_dept_unstacked = ______________.unstack()

print(avg_year_dept)

In [ ]:
# Step 3 — plot
_______________.plot(kind="____", marker="o", figsize=(10, 5))

plt.title("___")
plt.xlabel("Year")
plt.ylabel("Average Net Salary (euros)")
plt.xticks(rotation=___)
plt.legend(title="Department", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

# 6. Career Progression — Who Got Promoted?
To show names in the career path, we need to join the workers table now.

The fact table records which position each worker had each year.
If a worker has different `position_id` values across years — they were promoted.

In [ ]:
# Join workers — we need full_name and years_at_company now
# Hint: df.merge(other_df, on="shared_col", how="left")
hr = hr.merge(
    workers[["worker_id", "full_name", "years_at_company"]],
    on="__________",
    how="____"
)

print("After joining workers:")
print(hr[["full_name", "position_name", "year", "net_salary"]].head(5))

In [ ]:
# Count positions per worker — more than 1 means promoted
positions_per_worker = hr.groupby("worker_id")["position_id"].nunique()

promoted     = positions_per_worker[positions_per_worker > 1]
not_promoted = positions_per_worker[positions_per_worker == 1]

print("Workers promoted at least once:", len(promoted))
print("Workers with no promotion:     ", len(not_promoted))
print()
print("Promotions count:")
print(promoted.value_counts().sort_index())
print()
print("2 = promoted once, 3 = promoted twice, etc.")

### Your turn — top 3 most promoted workers

In [ ]:
# Hint: df.nlargest(n, series_name) — or sort and head
top_promoted = positions_per_worker.nlargest(___)

print("Most promoted workers:")
print(top_promoted.to_string())

### Your turn — career path of one specific worker

In [ ]:
# Pick a worker_id from the promoted list above
worker_id_to_check = "___"

# Filter — one specific worker
# Hint: df[df["col"] == variable]
career = hr[hr["worker_id"] == _______________]

# Show progression
print("Career path:")
print(career[["year", "position_name", "level", "base_salary", "net_salary"]].to_string(index=False))

In [ ]:
# See multiple workers at once — using a list of IDs
# Hint: df[df["col"].isin(["val1", "val2", "val3"])]
workers_to_check = ["___", "___", "___"]

careers = hr[hr["worker_id"].isin(_______________)]

# Sort so each worker's years appear together
# Hint: df.sort_values(["col1", "col2"])
careers_sorted = careers.sort_values(["__________", "____"])

print("Career paths — multiple workers:")
print(careers_sorted[["worker_id", "full_name", "year", "position_name", "level", "net_salary"]].to_string(index=False))

In [ ]:
# See a range of workers — W001 to W010
# Hint: df[df["col"].between("val1", "val2")] works on strings too
careers_range = hr[hr["worker_id"].between("____", "____")]

careers_range_sorted = careers_range.sort_values(["worker_id", "year"])

print("Career paths — W001 to W010:")
print(careers_range_sorted[["worker_id", "full_name", "year", "position_name", "net_salary"]].to_string(index=False))

# 7. Tenure Analysis — Do Workers Who Stay Longer Earn More?
Workers table already joined — `years_at_company` is available.

> *"Workers who stay longer might earn more — but is that because of tenure itself,
> or simply because they got promoted over time?
> Always check the confounding variable."*

In [ ]:
# Step 1 — create tenure groups using a function
# Same pattern as bag_size and price_tier — if / elif / else
def classify_tenure(years):
    if years <= ___:
        return "0-2 years"
    elif years <= ___:
        return "2-4 years"
    elif years <= ___:
        return "4-6 years"
    else:
        return "___"

# Apply to every row
# Hint: df["col"].apply(function_name)
hr["tenure_group"] = hr["_______________"].apply(_______________)

# Check distribution
# Hint: df["col"].value_counts().sort_index()
tenure_counts = hr["tenure_group"].value_counts().sort_index()

print("Workers per tenure group:")
print(_______________)

In [ ]:
# Step 2 — average salary per tenure group (naive first)
# Hint: df.groupby("col")["col"].mean().round(0)
avg_by_tenure = hr.groupby("____________")["__________"].______().round(0)

# Sort so groups appear in order
avg_by_tenure_sorted = avg_by_tenure.sort_index()

# Print
print("Average net salary by tenure group:")
print(_______________)
print()
print("Does longer tenure mean higher salary?")

In [ ]:
# Step 3 — but is this explained by position level?
# Workers who stay longer may simply be in higher positions
# Hint: df.groupby("col")["col"].mean().round(1)
avg_level_tenure = hr.groupby("____________")["_____"].______().round(1)

avg_level_tenure_sorted = avg_level_tenure.sort_index()

print("Average position level by tenure group:")
print(_______________)
print()
print("If longer tenure = higher level — the salary difference is about")
print("experience leading to promotion, not tenure itself")

### Your turn — fair comparison: same level, different tenure

**What to do:**
1. groupby `level` and `tenure_group` together
2. calculate average `net_salary` per group
3. sort by index so levels appear in order
4. read the output — for the same level, do different tenure groups earn differently

In [ ]:
# Hint: df.groupby(["col1","col2"])["col"].mean().round(0)
tenure_by_level = ___.____________(["_____", "____________"])["__________"].______().round(0)

tenure_by_level_sorted = tenure_by_level.sort_index()

print("Average net salary by level AND tenure group:")
print(_______________)
print()
print("Same level, different tenure — is there still a salary difference?")

In [ ]:
# Step 5 — visualise
_____________.______________(kind="bar", color="steelblue", edgecolor="white")

plt.title("Average Net Salary by Tenure Group")
plt.xlabel("Tenure Group")
plt.ylabel("Average Net Salary (euros)")
plt.xticks(rotation=___)

plt.tight_layout()
plt.show()

# 8. Challenges
Use the enriched `hr` table. Small hint for each one.

## Challenge 1 — Highest total earnings ever
Which worker earned the most in total across all years combined?
Show name, total net salary, and number of years in the dataset.

**Hint:** groupby `worker_id` and `full_name`, agg net_salary (sum) and year (nunique), then nlargest

In [ ]:
# Challenge 1 — which worker earned the most in total across all years?

# Step 1 — group by worker and calculate total earnings
# Hint: df.groupby(["col1","col2"]).agg(new_col=("col","function"))
total_by_worker = hr.groupby(["_________", "__________"]).agg(
    total_net_salary = ("__________", "___"),
    years_in_data    = ("____",       "________")
).round(0)

print("Total earnings per worker:")
print(total_by_worker.head(5))

In [ ]:
# Step 2 — find the top 5 using nlargest
# Hint: df.nlargest(n, "col")
top_earners = _______________.____________(___, "________________")

print("Top 5 total earners:")
print(top_earners.to_string())

## Challenge 2 — Department with highest average bonus
Which department pays the highest average bonus?
Is it explained by position level?

**Hint:** groupby department, mean bonus AND mean level — compare both rankings

In [ ]:
# Challenge 2 — which department pays the highest average bonus?
# And is it explained by position level?

# Step 1 — average bonus per department
# Hint: df.groupby("col")["col"].mean().round(0)
bonus_by_dept = hr.groupby("_______________")["_____"].______().round(0)

print("Average bonus by department:")
print(_______________)

In [ ]:
# Step 2 — average position level per department
# Hint: df.groupby("col")["col"].mean().round(1)
level_by_dept = hr.groupby("_______________")["_____"].______().round(1)

print("Average level by department:")
print(_______________)

In [ ]:
# Step 3 — combine both into one summary table
# Hint: pd.DataFrame({"col1": series1, "col2": series2})
summary = pd.DataFrame({
    "avg_bonus": _______________,
    "avg_level": _______________
})

# Step 4 — sort by bonus highest first
# Hint: df.sort_values("col", ascending=False)
summary_sorted = summary.sort_values("___________", ascending=_______)

# Step 5 — print
print("Bonus and level by department:")
print(_______________)
print()
print("Does the department with the highest bonus also have the highest level?")
print("If yes — the bonus difference is explained by seniority, not the department")

## Challenge 3 — Hires per year
How many workers were hired each year?
Plot a bar chart.

**Hint:** use `workers["admission_date"]` — convert to datetime, extract year, value_counts

In [ ]:
# Challenge 3 — how many workers were hired each year?

# Step 1 — convert admission_date to datetime
# Hint: pd.to_datetime(df["col"])
workers["admission_date"] = pd.to_datetime(workers["_______________"])

print("Date column converted:")
print(workers["admission_date"].head(5))

In [ ]:
# Step 2 — extract the year from admission_date
# Hint: df["date_col"].dt.year
workers["admission_year"] = workers["_______________"].dt.____

print("Admission year — first 5 rows:")
print(workers[["full_name", "admission_date", "admission_year"]].head(5))

In [ ]:
# Step 3 — count workers hired per year
# Hint: df["col"].value_counts().sort_index()
hires_per_year =

print("Hires per year:")
print(_______________)

In [ ]:
# Step 4 — plot a bar chart
# Hint: df.plot(kind="bar", color="steelblue")
____________.plot(kind="___", color="steelblue")

plt.title("___")
plt.xlabel("Year")
plt.ylabel("___")
plt.xticks(rotation=___)

plt.tight_layout()
plt.show()

print()
print("Which year had the most hires? ___")
print("Which year had the fewest? ___")

# Answers — only open after you tried!

In [ ]:
# Challenge 1 — highest total earnings
total_by_worker = hr.groupby(["worker_id", "full_name"]).agg(
    total_net_salary = ("net_salary", "sum"),
    years_in_data    = ("year",       "nunique")
).round(0)

top_earner = total_by_worker.nlargest(5, "total_net_salary")
print("Top 5 total earners:")
print(top_earner.to_string())

In [ ]:
# Challenge 2 — bonus by department
bonus_by_dept = hr.groupby("department_name")["bonus"].mean().round(0)
level_by_dept = hr.groupby("department_name")["level"].mean().round(1)

summary = pd.DataFrame({
    "avg_bonus": bonus_by_dept,
    "avg_level": level_by_dept
}).sort_values("avg_bonus", ascending=False)

print("Bonus and level by department:")
print(summary.to_string())

In [ ]:
# Challenge 3 — hires per year
workers["admission_date"] = pd.to_datetime(workers["admission_date"])
workers["admission_year"] = workers["admission_date"].dt.year

hires = workers["admission_year"].value_counts().sort_index()

hires.plot(kind="bar", color="steelblue")
plt.title("Workers Hired per Year")
plt.xlabel("Year")
plt.ylabel("Number of Hires")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(hires.to_string())